# Structural Connectome A

EC2-native early pipeline notebook for Steps 1 through 4.

This notebook assumes the canonical EC2 layout:
- `~/exp/data/Images/dti`
- `~/exp/data/Images/mri`
- `~/exp/data/derivatives`

Steps 1-4 now use guarded helper modules with dry-run, target filtering, and provenance/QC exports before any heavy command is launched.


In [ ]:
from pathlib import Path
import importlib
import importlib.util
import os
import sys

PROJECT_ROOT = Path.home() / "exp"
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

for candidate in [
    Path.home() / "bin",
    Path.home() / "mrtrix3" / "bin",
    Path.home() / "fsl" / "bin",
    Path.home() / "fsl" / "share" / "fsl" / "bin",
]:
    if candidate.exists() and str(candidate) not in os.environ.get("PATH", "").split(os.pathsep):
        os.environ["PATH"] = f"{candidate}{os.pathsep}" + os.environ.get("PATH", "")

os.environ.setdefault("FSLDIR", str(Path.home() / "fsl"))

from connectome_pipeline.pipeline_paths import resolve_pipeline_paths

paths = resolve_pipeline_paths(create_layout=True)
print(paths.format_summary())

from connectome_pipeline import connectome_run_control
connectome_run_control = importlib.reload(connectome_run_control)

DRY_RUN = True
EXECUTE = False
FORCE_INVALID_ONLY = True
INCLUDE = []
EXCLUDE = []
BACKUP_BEFORE_REPLACE = True
VALIDATION_GATE = True
INCLUDE_WARN_QC = False
# Safety guard: global EXECUTE=True should not launch Eddy unless this is explicit.
RUN_EDDY_EXECUTION_AFTER_PREFLIGHT = False

RUN_CONTROL = connectome_run_control.RunControlConfig.from_values(
    dry_run=DRY_RUN,
    execute=EXECUTE,
    force_invalid_only=FORCE_INVALID_ONLY,
    include=INCLUDE,
    exclude=EXCLUDE,
    backup_before_replace=BACKUP_BEFORE_REPLACE,
    validation_gate=VALIDATION_GATE,
    include_warn=INCLUDE_WARN_QC,
)
connectome_run_control.print_run_control(RUN_CONTROL)


def module_status(*names):
    return {name: importlib.util.find_spec(name) is not None for name in names}


def visible_children(path: Path, limit: int = 10):
    if not path.exists():
        return []
    return [p.name for p in sorted(path.iterdir()) if not p.name.startswith('.')][:limit]


## Pipeline Inventory Table

Run this cell first to see the EC2 pipeline completeness snapshot across the intermediate stages. It uses the current contents of `~/exp/data/derivatives` and helps show which stage outputs are still missing.


In [ ]:
import importlib
from connectome_pipeline import pipeline_status

pipeline_status = importlib.reload(pipeline_status)

status_locations = ["ec2", "s3"]

for status_location in status_locations:
    pipeline_status.display_all_stage_group_status(
        paths.deriv_root,
        cohort_dti_csv=paths.cohort_dti_csv,
        aal_mni=paths.aal_mni,
        location=status_location,
        s3_root="s3://sabeesh/exp",
        return_status=False,
    )


## Step 1: Raw DWI to MIF

This step becomes runnable once raw DWI data is pushed into `~/exp/data/Images/dti` and `dwi_convert.py` is available in `~/exp`.


In [ ]:

step1_modules = module_status("dwi_convert")
raw_dwi_children = visible_children(paths.raw_dwi_root)
print(f"raw_dwi_root exists      : {paths.raw_dwi_root.exists()}")
print(f"raw_dwi_root sample      : {raw_dwi_children}")
print(f"dwi_convert.py available : {step1_modules['dwi_convert']}")
if not step1_modules["dwi_convert"]:
    print("Step 1 is gated until dwi_convert.py is copied into ~/exp.")
else:
    from connectome_pipeline import dwi_convert
    dwi_convert = importlib.reload(dwi_convert)

    STEP1_STRICT_RAW_DWI_ONLY = True
    STEP1_MIN_DW_DIRECTIONS = dwi_convert.DEFAULT_MIN_DW_DIRECTIONS
    STEP1_SOURCE_MANIFEST = paths.deriv_root / "qc" / "dwi_source_manifest.csv"

    discovered_series, discovery_counts = dwi_convert.find_all_series(paths.raw_dwi_root)
    source_manifest = dwi_convert.write_source_manifest(discovered_series, STEP1_SOURCE_MANIFEST)
    filtered_series, source_gate_counts, rejected_sources = dwi_convert.filter_strict_raw_dwi_series(
        discovered_series,
        strict_raw_dwi_sources=STEP1_STRICT_RAW_DWI_ONLY,
    )

    print("Step 1 source contract")
    print("----------------------")
    print(f"discovered series      : {discovery_counts}")
    print(f"strict raw DWI only    : {STEP1_STRICT_RAW_DWI_ONLY}")
    print(f"min DW directions      : {STEP1_MIN_DW_DIRECTIONS}")
    print(f"source manifest        : {STEP1_SOURCE_MANIFEST}")
    print(f"source gate counts     : {dict(source_gate_counts)}")
    print(f"rejected raw candidates: {len(rejected_sources)}")
    if rejected_sources:
        print("First rejected raw candidates:")
        for row in rejected_sources[:10]:
            print(f"  {row['series_path']} | {row['source_status']} | {row['source_reason']}")

    STEP1_CONVERSION_CFG = {
        "all_series": filtered_series,
        "out_dwi": paths.deriv_root / "mif_dwi",
        "out_derived": paths.deriv_root / "mif_derived",
        "force": bool(RUN_CONTROL.execute and not RUN_CONTROL.force_invalid_only),
        "dry_run": RUN_CONTROL.effective_dry_run,
        "jobs": 1,
        "min_dw_directions": STEP1_MIN_DW_DIRECTIONS,
        "strict_raw_dwi_sources": STEP1_STRICT_RAW_DWI_ONLY,
        "source_manifest_csv": STEP1_SOURCE_MANIFEST,
    }
    print()
    print("Step 1 conversion config is prepared but not launched by this setup cell.")
    print("Run dwi_convert.run_conversion_batch(**STEP1_CONVERSION_CFG) only after reviewing the manifest.")


## Steps 2 and 3: Denoise and Gibbs

This cell uses the shared run-control settings to validate or run `dwidenoise` and `mrdegibbs`. It preserves the required file contract for Eddy: `mif_unringed/<series>_den_unr.mif`. By default it is a dry-run/status pass and only targets missing or invalid outputs.


In [ ]:
import importlib
import os

from connectome_pipeline import dwi_denoise_gibbs
from connectome_pipeline import pipeline_status

dwi_denoise_gibbs = importlib.reload(dwi_denoise_gibbs)
pipeline_status = importlib.reload(pipeline_status)

STEP23_CFG = {
    "deriv_root": paths.deriv_root,
    "jobs": min(4, max(1, os.cpu_count() or 1)),
    "force": bool(RUN_CONTROL.execute and not RUN_CONTROL.force_invalid_only),
    "dry_run": RUN_CONTROL.effective_dry_run,
    "force_invalid_only": RUN_CONTROL.force_invalid_only,
    "include": RUN_CONTROL.include,
    "exclude": RUN_CONTROL.exclude,
    "run_denoise": True,
    "run_gibbs": True,
    "summary_csv": paths.deriv_root / "qc" / "denoise_gibbs_summary.csv",
}

print("Denoise/Gibbs configuration")
print("-----------------------------")
for key, value in STEP23_CFG.items():
    print(f"{key:<20}: {value}")

step23_result = dwi_denoise_gibbs.run_denoise_gibbs_batch(**STEP23_CFG)
print("\nDenoise/Gibbs result")
print("---------------------")
for key, value in step23_result.items():
    print(f"{key:<20}: {value}")

print("\nCurrent stage status")
status = pipeline_status.display_group_stage_status(
    paths.deriv_root,
    cohort_dti_csv=paths.cohort_dti_csv,
    title="Pipeline group status (through Eddy)",
)
print()
pipeline_status.print_derivatives_root_diagnostics(paths.deriv_root)
print()
for stage, counts in status["stage_artifact_counts"].items():
    print(
        f"{stage:<10} matched={counts['matched_paths']} | "
        f"usable={counts['usable_existing_paths']} | "
        f"broken_symlinks={counts['broken_symlink_paths']}"
    )

step23_result


## Step 4: Eddy

The config cell below is EC2-native and points only at `~/exp/data/derivatives`. By default it stays in preflight mode so running the next cell will validate the job plan without launching a full batch until you flip the flag.


In [ ]:
# (base) bash-5.2$ source ~/exp/eddy_env.sh
# (base) bash-5.2$ eddy_run 1 1 all

In [ ]:
import importlib
import os
from pathlib import Path

from connectome_pipeline import dwi_eddy
from connectome_pipeline import pipeline_status


dwi_eddy = importlib.reload(dwi_eddy)
pipeline_status = importlib.reload(pipeline_status)

EDDY_CFG = {
    "deriv_root": paths.deriv_root,
    "cohort_dti_csv": paths.cohort_dti_csv,
    "stage_root": Path("/scratch/eddy_stage") if Path("/scratch").exists() else Path("/tmp/eddy_stage"),
    "jobs": 1,
    "threads_per_job": min(4, max(1, os.cpu_count() or 4)),
    "force": bool(RUN_CONTROL.execute and not RUN_CONTROL.force_invalid_only),
    "pe_dir": "j-",  # supervisor-compatible ADNI default; header mismatch is still logged by dwi_eddy
    "eddy_options": "--slm=linear --data_is_shelled",
    "run_eddy_qc": False,
    "preflight_only": (not RUN_EDDY_EXECUTION_AFTER_PREFLIGHT) or RUN_CONTROL.effective_dry_run,
    "b0_threshold": dwi_eddy.DEFAULT_B0_THRESHOLD,
    "allow_cpu_fallback": False,
    "allow_input_fallback": False,
}

print("Eddy configuration")
print("------------------")
for key, value in EDDY_CFG.items():
    print(f"{key:<16}: {value}")
print()
pipeline_status.print_eddy_status(paths.deriv_root)
pending_ad = pipeline_status.collect_pending_eddy_series(
    paths.deriv_root,
    cohort_dti_csv=paths.cohort_dti_csv,
    group="ad",
)
print(f"Pending AD Eddy series: {len(pending_ad)}")
pending_ad[:20]


In [ ]:
eddy_result = dwi_eddy.run_eddy_preproc(
    deriv_root=EDDY_CFG["deriv_root"],
    jobs=EDDY_CFG["jobs"],
    threads_per_job=EDDY_CFG["threads_per_job"],
    force=EDDY_CFG["force"],
    stage_root=EDDY_CFG["stage_root"],
    pe_dir=EDDY_CFG["pe_dir"],
    eddy_options=EDDY_CFG["eddy_options"],
    cohort_dti_csv=EDDY_CFG["cohort_dti_csv"],
    run_eddy_qc=EDDY_CFG["run_eddy_qc"],
    preflight_only=EDDY_CFG["preflight_only"],
    b0_threshold=EDDY_CFG["b0_threshold"],
    allow_cpu_fallback=EDDY_CFG["allow_cpu_fallback"],
    allow_input_fallback=EDDY_CFG["allow_input_fallback"],
)
eddy_status_after = pipeline_status.print_eddy_status(paths.deriv_root)
print(f"Eddy summary CSV      : {eddy_result.get('summary_csv')}")
print(f"B0 provenance CSV     : {eddy_result.get('b0_provenance_csv')}")
print(f"Eddy quarantine CSV   : {eddy_result.get('quarantine_csv')}")
eddy_result
